# Evidence-First LLM Analysis: πολιτεία in Philo of Alexandria

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Purpose and place in the tutorial series</a>
* <a href="#architecture">2 - Understand the evidence-first research architecture</a>
* <a href="#boundaries">3 - Define the research question and epistemic boundaries</a>
* <a href="#setup">4 - Install dependencies and load the local server</a>
* <a href="#configuration">5 - Configure OpenRouter without blocking local research</a>
* <a href="#model-check">6 - Verify the selected model</a>
* <a href="#helpers">7 - Build MCP, text, and JSON helpers</a>
* <a href="#live-schemas">8 - Verify the live evidence-tool schemas</a>
* <a href="#philo-scope">9 - Verify the Philo textgroup scope</a>
* <a href="#search-plan">10 - Design complementary search strategies</a>
* <a href="#search-run">11 - Run bounded, paged searches</a>
* <a href="#normalize-candidates">12 - Normalize and merge search candidates</a>
* <a href="#coverage">13 - Inspect source overlap and work distribution</a>
* <a href="#sampling-policy">14 - Define a bounded stratified sampling policy</a>
* <a href="#sample-candidates">15 - Select candidate passages across works</a>
* <a href="#fetch-contexts">16 - Retrieve passage text and center context on the target stem</a>
* <a href="#quality-audit">17 - Audit retrieval and lexical relevance</a>
* <a href="#baseline">18 - Build a descriptive baseline before interpretation</a>
* <a href="#evidence-packet">19 - Assemble the research evidence packet</a>
* <a href="#packet-validation">20 - Validate packet provenance and size</a>
* <a href="#analysis-contract">21 - Define the structured analysis contract</a>
* <a href="#openrouter-client">22 - Build a defensive OpenRouter JSON helper</a>
* <a href="#synthesis">23 - Optionally request a cited synthesis</a>
* <a href="#citation-audit">24 - Audit model citations deterministically</a>
* <a href="#render-analysis">25 - Render the structured analysis</a>
* <a href="#skeptical-review">26 - Optionally run a skeptical review</a>
* <a href="#revision">27 - Optionally produce an evidence-constrained revision</a>
* <a href="#interpretive-cautions">28 - Interpret the result cautiously</a>
* <a href="#follow-up">29 - Extend the research design</a>
* <a href="#privacy-cost">30 - Privacy, cost, and reproducibility</a>
* <a href="#troubleshooting">31 - Troubleshooting reference</a>
* <a href="#next-steps">32 - Continue learning</a>
* <a href="#sources">33 - Sources</a>
* <a href="#required-libraries">34 - Required libraries</a>
* <a href="#notebook-version">35 - Notebook version</a>

## 1 - Purpose and place in the tutorial series <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook is the capstone of the tutorial series. It combines scoped Scaife search, passage retrieval, explicit sampling, provenance checks, and optional OpenRouter synthesis in a reproducible case study: **how does Philo of Alexandria use the lexical family around `πολιτεία`?**

The preceding notebooks supply each layer:

- notebooks `01_`–`04_` establish CTS, Scaife, MCP, search, and URN navigation;
- notebook `05_` documents the complete tool catalog;
- notebook `06_` explains guarded LLM/MCP interaction;
- notebook `07_` is the advanced `search_perseus` reference;
- notebook `08_` explains reader search, Scaife passage retrieval, caching, and cross-service evidence packets.

This notebook does not ask an LLM to search autonomously. Python first constructs and validates a bounded evidence packet. Only that reviewed packet is optionally sent to OpenRouter for synthesis.

By the end, you should be able to:

1. design complementary lemma, wildcard, and surface-form searches;
2. collect a bounded number of search pages rather than silently treating page 1 as the corpus;
3. deduplicate passages while preserving which strategies found them;
4. inspect work-level coverage before sampling;
5. select a stratified sample instead of taking only the first ranked hits;
6. retrieve context windows centered on the relevant lexical stem;
7. distinguish search failure, retrieval failure, and lexical irrelevance;
8. constrain model output to a machine-checkable structure;
9. reject citations that are absent from the evidence packet;
10. use critique and revision as separate, inspectable stages.

> Local evidence collection requires live Scaife access but no OpenRouter key. All credentialed model calls are disabled by default.

## 2 - Understand the evidence-first research architecture <a class="anchor" id="architecture"></a>
##### [Back to ToC](#TOC)

The workflow separates retrieval from interpretation:

```text
Research question
      |
      v
Verify Philo scope in Scaife
      |
      v
Run complementary, bounded search strategies
      |
      v
Normalize -> deduplicate -> measure work coverage
      |
      v
Apply explicit stratified sampling policy
      |
      v
Retrieve passage text -> center context -> quality audit
      |
      v
Validated evidence packet
      |
      +------------------------------+
      |                              |
      v                              v
Human inspection                Optional OpenRouter JSON synthesis
                                     |
                                     v
                              deterministic citation audit
                                     |
                           optional critique and revision
```

The LLM never determines what entered the evidence packet. It cannot add new passages through tool calls, and its cited URNs are checked against the packet after generation.

This architecture reduces two common failure modes:

- **retrieval opacity:** a fluent answer conceals which passages were actually examined;
- **citation drift:** the model cites a real-looking but unsupplied URN or attaches a supplied URN to an unsupported claim.

The second problem cannot be solved completely by automatic citation membership checks, which verify provenance rather than semantic entailment. That is why the workflow keeps a separate skeptical-review stage.

## 3 - Define the research question and epistemic boundaries <a class="anchor" id="boundaries"></a>
##### [Back to ToC](#TOC)

Research question:

> **What semantic and metaphorical uses of the `πολιτεία` lexical family are visible in a bounded sample of Scaife-indexed Greek passages attributed to Philo of Alexandria?**

This wording is intentionally narrower than “What does `πολιτεία` mean in Philo?” The notebook does not establish a complete lexicon entry or a statistically representative corpus study.

What the workflow can support:

- observations about retrieved Greek passages;
- tentative grouping of uses visible in the sample;
- comparison of work distribution and search-strategy overlap;
- explicit identification of missing evidence and follow-up searches.

What it cannot establish by itself:

- exhaustive frequency across the entire Philonic corpus;
- authoritative lemmatization or morphology;
- complete textual context beyond the retrieved passage units;
- a critical-edition judgment when passage text includes apparatus or editorial material;
- historical originality relative to Plato, Aristotle, Stoicism, Josephus, or other corpora;
- a translation, because the evidence packet supplies Greek text only;
- semantic support merely because a model cites an allowed URN.

The primary evidence is Scaife-indexed Greek text identified by Scaife passage URNs. The generated interpretation is secondary analysis.

## 4 - Install dependencies and load the local server <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

The first cell installs Perseus MCP into the active Jupyter kernel. It defaults to the local repository in editable mode for development-branch work; change the switch to `"pypi"` for the published package. The setup then locates the repository root, configures the shared metadata cache, loads `.env` without overriding existing environment variables, reloads `perseus_mcp.server`, and obtains the in-process FastMCP server object.

Set `PERSEUS_MCP_INSTALL_SOURCE` in the install cell to `"repo"` for editable development-branch work or `"pypi"` for the published package.

The OpenRouter key is deliberately not requested here. Every evidence-collection cell can run first.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Use "repo" while working on this development checkout.
# Use "pypi" to run against the published package normal users install.
PERSEUS_MCP_INSTALL_SOURCE = "repo"  # "repo" or "pypi"
PERSEUS_MCP_PYPI_SPEC = "perseus-mcp"

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

install_source = PERSEUS_MCP_INSTALL_SOURCE.lower()
if install_source == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find the Perseus-mcp repository from {START}. Open this notebook inside the repository checkout or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    package_target = ["--editable", str(REPO_ROOT)]
    package_label = f"editable repository at {REPO_ROOT}"
elif install_source == "pypi":
    package_target = ["--force-reinstall", PERSEUS_MCP_PYPI_SPEC]
    package_label = PERSEUS_MCP_PYPI_SPEC
else:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *package_target,
        "python-dotenv>=1.0.0",
    ]
)

print(f"Installed perseus-mcp from {package_label} into this kernel")

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
import html
import importlib
import json
import os
import re
import sys
import unicodedata

import httpx
from dotenv import load_dotenv
from fastmcp import Client
from IPython.display import Markdown, display

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

PERSEUS_MCP_INSTALL_SOURCE = globals().get("PERSEUS_MCP_INSTALL_SOURCE", "repo").lower()
if PERSEUS_MCP_INSTALL_SOURCE not in {"repo", "pypi"}:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

if PERSEUS_MCP_INSTALL_SOURCE == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    SRC_DIR = REPO_ROOT / "src"
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
elif REPO_ROOT is not None:
    SRC_DIR = REPO_ROOT / "src"
    src_dir_resolved = SRC_DIR.resolve()

    def _is_repo_src(path_entry):
        try:
            return Path(path_entry).resolve() == src_dir_resolved
        except (OSError, RuntimeError):
            return False

    sys.path = [path_entry for path_entry in sys.path if not _is_repo_src(path_entry)]

if "load_dotenv" in globals():
    if REPO_ROOT is not None:
        load_dotenv(REPO_ROOT / ".env", override=False)
    else:
        load_dotenv(override=False)

CACHE_ROOT = REPO_ROOT if REPO_ROOT is not None else START
os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(CACHE_ROOT / ".cache" / "perseus-mcp"),
)

for module_name in [
    name
    for name in list(sys.modules)
    if name == "perseus_mcp" or name.startswith("perseus_mcp.")
]:
    del sys.modules[module_name]

from fastmcp import Client
from perseus_mcp import server

server = importlib.reload(server)
mcp = server.mcp

print(f"Install source: {PERSEUS_MCP_INSTALL_SOURCE}")
print(f"Repository root: {REPO_ROOT if REPO_ROOT is not None else 'not found'}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server: {mcp.name}")

## 5 - Configure OpenRouter without blocking local research <a class="anchor" id="configuration"></a>
##### [Back to ToC](#TOC)

Copy `.env.example` to `.env` and set:

```dotenv
OPENROUTER_API_KEY=sk-or-v1-...
```

Optional settings:

```dotenv
OPENROUTER_MODEL=openrouter/free
OPENROUTER_APP_URL=https://github.com/tonyjurg/Perseus-mcp
OPENROUTER_APP_NAME=Perseus MCP Philo Analysis
```

The key is loaded or securely prompted for only when an optional model call is enabled. It is never placed in the message history or evidence packet.

`openrouter/free` is OpenRouter's Free Models Router rather than a fixed model. It chooses among free models available at request time and filters for capabilities requested by the call, such as structured output. We prefer it here for flexibility and to avoid breaking the notebook when one selected free model is removed, renamed, or temporarily unavailable. The tradeoff is that separate runs may use different concrete models, so every synthesis records OpenRouter's `resolved_model`. Set `OPENROUTER_MODEL` to a fixed slug for experiments that require exact model reproducibility.

In [4]:
OPENROUTER_CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODELS_URL = "https://openrouter.ai/api/v1/models"
OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL",
    "openrouter/free",
)
OPENROUTER_APP_URL = os.getenv("OPENROUTER_APP_URL")
OPENROUTER_APP_NAME = os.getenv(
    "OPENROUTER_APP_NAME",
    "Perseus MCP Philo Analysis",
)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


def require_openrouter_api_key():
    global OPENROUTER_API_KEY
    if not OPENROUTER_API_KEY:
        OPENROUTER_API_KEY = getpass("OpenRouter API key: ").strip()
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY is required for model calls.")
    return OPENROUTER_API_KEY


def openrouter_headers():
    headers = {
        "Authorization": f"Bearer {require_openrouter_api_key()}",
        "Content-Type": "application/json",
    }
    if OPENROUTER_APP_URL:
        headers["HTTP-Referer"] = OPENROUTER_APP_URL
    if OPENROUTER_APP_NAME:
        headers["X-OpenRouter-Title"] = OPENROUTER_APP_NAME
    return headers


print(f"Configured model: {OPENROUTER_MODEL}")
print(f"API key already loaded: {bool(OPENROUTER_API_KEY)}")

Configured model: openrouter/free
API key already loaded: False


## 6 - Verify the selected model <a class="anchor" id="model-check"></a>
##### [Back to ToC](#TOC)

This notebook uses ordinary chat completion rather than model-driven tool calling. For a fixed model, the public catalog is useful for checking:

- whether the configured model ID exists;
- context length;
- current pricing fields;
- whether `response_format` is advertised for optional JSON mode.

For `openrouter/free`, the check describes the router contract instead: the concrete free model and its context length are selected when the request is made. Requesting `response_format` lets the router filter for structured-output support. A successful check does not guarantee provider capacity. The evidence packet size is measured later and must remain conservative because router-selected context windows can vary.

In [5]:
async def get_openrouter_model_info(model_id):
    if model_id == "openrouter/free":
        return {
            "id": model_id,
            "name": "OpenRouter Free Models Router",
            "available": True,
            "router": True,
            "pricing": {"prompt": "0", "completion": "0"},
            "supported_parameters": ["response_format", "structured_outputs"],
            "note": (
                "The concrete model is selected at request time from free "
                "models compatible with the requested features."
            ),
        }

    async with httpx.AsyncClient(timeout=30.0) as http:
        response = await http.get(OPENROUTER_MODELS_URL)
        response.raise_for_status()
        models = response.json().get("data", [])

    model = next((item for item in models if item.get("id") == model_id), None)
    if model is None:
        return {"id": model_id, "available": False}

    return {
        "id": model.get("id"),
        "name": model.get("name"),
        "available": True,
        "context_length": model.get("context_length"),
        "pricing": model.get("pricing"),
        "supported_parameters": sorted(model.get("supported_parameters") or []),
    }


try:
    selected_model_info = await get_openrouter_model_info(OPENROUTER_MODEL)
    print(json.dumps(selected_model_info, ensure_ascii=False, indent=2))
except httpx.HTTPError as exc:
    selected_model_info = None
    print(f"Could not verify the OpenRouter catalog: {type(exc).__name__}: {exc}")

{
  "id": "openrouter/free",
  "name": "OpenRouter Free Models Router",
  "available": true,
  "router": true,
  "pricing": {
    "prompt": "0",
    "completion": "0"
  },
  "supported_parameters": [
    "response_format",
    "structured_outputs"
  ],
  "note": "The concrete model is selected at request time from free models compatible with the requested features."
}


## 7 - Build MCP, text, and JSON helpers <a class="anchor" id="helpers"></a>
##### [Back to ToC](#TOC)

The helpers below support four tasks:

- parse MCP JSON/plaintext results;
- normalize Scaife result records into stable candidate fields;
- fold Greek accents while preserving an index map into the original string;
- extract JSON and CTS URNs from model output.

The accent-folding index map lets the notebook find `πολιτει` across polytonic forms and then cut a context window from the original accented passage text.

In [9]:
TAG_RE = re.compile(r"<[^>]+>")
CTS_URN_RE = re.compile(r"urn:cts:[^\s\]\[(){}<>\"'`]+")
GREEK_TOKEN_RE = re.compile(r"[Ͱ-Ͽἀ-῿]+")
TARGET_FOLDED_STEM = "πολιτει"


def tool_text(result):
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


def clean_snippet(value):
    if isinstance(value, list):
        value = " ".join(str(item) for item in value)
    return " ".join(html.unescape(TAG_RE.sub("", value or "")).split())


def unique_nonempty(values):
    seen = set()
    output = []
    for value in values:
        if value and value not in seen:
            seen.add(value)
            output.append(value)
    return output


def fold_greek(text):
    return "".join(
        character.casefold()
        for character in unicodedata.normalize("NFD", text or "")
        if not unicodedata.combining(character)
    )


def folded_text_with_original_indexes(text):
    folded_characters = []
    original_indexes = []
    for original_index, character in enumerate(text or ""):
        for decomposed in unicodedata.normalize("NFD", character):
            if unicodedata.combining(decomposed):
                continue
            for folded_character in decomposed.casefold():
                folded_characters.append(folded_character)
                original_indexes.append(original_index)
    return "".join(folded_characters), original_indexes


def centered_context(text, folded_stem=TARGET_FOLDED_STEM, max_chars=2800):
    folded, index_map = folded_text_with_original_indexes(text)
    match_index = folded.find(folded_stem)
    if match_index < 0 or not index_map:
        clipped = len(text) > max_chars
        return text[:max_chars], False, clipped

    original_match = index_map[match_index]
    half = max_chars // 2
    start = max(0, original_match - half)
    end = min(len(text), start + max_chars)
    start = max(0, end - max_chars)
    prefix = "...[context begins mid-passage]\n" if start else ""
    suffix = "\n...[context ends mid-passage]" if end < len(text) else ""
    return prefix + text[start:end] + suffix, True, start > 0 or end < len(text)


def extract_json_object(text):
    stripped = (text or "").strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```(?:json)?\s*", "", stripped, count=1)
        stripped = re.sub(r"\s*```$", "", stripped, count=1)
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start < 0 or end <= start:
            raise
        return json.loads(stripped[start : end + 1])


def extract_cts_urns(value):
    if not isinstance(value, str):
        value = json.dumps(value, ensure_ascii=False)
    return sorted({match.group(0).rstrip(".,;:") for match in CTS_URN_RE.finditer(value)})

## 8 - Verify the live evidence-tool schemas <a class="anchor" id="live-schemas"></a>
##### [Back to ToC](#TOC)

The evidence pipeline relies on four MCP tools. This drift check verifies their current argument sets before any research requests are made.

In [10]:
EXPECTED_EVIDENCE_SCHEMAS = {
    "search_perseus": {
        "query", "language", "query_format", "author", "search_kind",
        "preserve_operators", "page_num", "text_group", "work", "result_format",
    },
    "get_scaife_library_metadata": {"urn"},
    "get_scaife_passage_json": {"urn"},
    "get_scaife_passage_text": {"urn"},
}

async with Client(mcp) as client:
    live_tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in live_tools}
schema_drift = {}
for name, expected in EXPECTED_EVIDENCE_SCHEMAS.items():
    if name not in tool_by_name:
        schema_drift[name] = {"missing_tool": True}
        continue
    actual = set((tool_by_name[name].inputSchema or {}).get("properties", {}))
    if actual != expected:
        schema_drift[name] = {
            "missing": sorted(expected - actual),
            "unexpected": sorted(actual - expected),
        }

print(json.dumps(schema_drift, indent=2))
assert not schema_drift, "Evidence-tool schemas have drifted from this notebook"

{}


## 9 - Verify the Philo textgroup scope <a class="anchor" id="philo-scope"></a>
##### [Back to ToC](#TOC)

The research constant is the Scaife textgroup `urn:cts:greekLit:tlg0018`. It is verified through Scaife library metadata before use.

The Perseus CTS capabilities feed used by local discovery may not advertise the same textgroup. That absence does not invalidate a Scaife scope; it demonstrates why service-specific discovery matters.

The notebook therefore:

1. validates the Scaife metadata URN and label;
2. records the number of Scaife works/resources exposed beneath it;
3. checks CTS author-name discovery only as a comparison;
4. uses explicit `text_group=PHILO_TEXTGROUP` for Scaife search.

In [11]:
PHILO_TEXTGROUP = "urn:cts:greekLit:tlg0018"

async with Client(mcp) as client:
    philo_metadata = await call_json(
        client,
        "get_scaife_library_metadata",
        {"urn": PHILO_TEXTGROUP},
    )
    cts_philo_candidates = await call_json(
        client,
        "find_author_names",
        {"query": "Philo", "language": "greek", "limit": 20},
    )

if philo_metadata.get("urn") != PHILO_TEXTGROUP:
    raise RuntimeError("Scaife metadata did not confirm the requested Philo textgroup URN.")

philo_label = (
    philo_metadata.get("label")
    or philo_metadata.get("title")
    or philo_metadata.get("name")
)
cts_exact_matches = [
    author for author in cts_philo_candidates.get("authors", [])
    if author.get("urn") == PHILO_TEXTGROUP
]

print(
    json.dumps(
        {
            "scaife_textgroup": PHILO_TEXTGROUP,
            "scaife_label": philo_label,
            "scaife_work_count": len(philo_metadata.get("works") or []),
            "cts_name_candidate_count": cts_philo_candidates.get("match_count"),
            "cts_exact_textgroup_match": bool(cts_exact_matches),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scaife_textgroup": "urn:cts:greekLit:tlg0018",
  "scaife_label": "Philo Judaeus",
  "scaife_work_count": 31,
  "cts_name_candidate_count": 3,
  "cts_exact_textgroup_match": false
}


## 10 - Design complementary search strategies <a class="anchor" id="search-plan"></a>
##### [Back to ToC](#TOC)

No single search is treated as exhaustive. The plan uses three complementary strategies:

| Strategy | Query | Purpose | Limitation |
|---|---|---|---|
| Lemma | `πολιτεία` with `search_kind="lemma"` | Ask the morphological index for the headword | A zero result can reflect indexing coverage rather than lexical absence |
| Wildcard form | `πολιτει*` | Capture visible forms beginning with the accent-folded stem | Operator/wildcard semantics are upstream and may be broader or narrower than expected |
| Surface variants | OR-style list of common inflections | Sensitivity check using explicitly named forms | Not a complete paradigm and dependent on upstream OR behavior |

All searches are server-scoped to the Philo textgroup and use `instances` so highlighted snippets remain available.

Each strategy is bounded to a small maximum number of pages. The result is a candidate pool, not a claim to have searched every possible occurrence.

In [12]:
SEARCH_PLAN = [
    {
        "id": "lemma_politeia",
        "label": "lemma: πολιτεία",
        "arguments": {
            "query": "πολιτεία",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
            "text_group": PHILO_TEXTGROUP,
            "result_format": "instances",
        },
    },
    {
        "id": "wildcard_politei",
        "label": "form wildcard: πολιτει*",
        "arguments": {
            "query": "πολιτει*",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "preserve_operators": True,
            "text_group": PHILO_TEXTGROUP,
            "result_format": "instances",
        },
    },
    {
        "id": "surface_variants",
        "label": "explicit surface variants",
        "arguments": {
            "query": "πολιτεία | πολιτείας | πολιτείᾳ | πολιτείαν | πολιτεῖαι | πολιτειῶν | πολιτείαις",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "preserve_operators": True,
            "text_group": PHILO_TEXTGROUP,
            "result_format": "instances",
        },
    },
]

MAX_PAGES_PER_STRATEGY = 3
print(json.dumps(SEARCH_PLAN, ensure_ascii=False, indent=2))

[
  {
    "id": "lemma_politeia",
    "label": "lemma: πολιτεία",
    "arguments": {
      "query": "πολιτεία",
      "language": "greek",
      "query_format": "unicode",
      "search_kind": "lemma",
      "text_group": "urn:cts:greekLit:tlg0018",
      "result_format": "instances"
    }
  },
  {
    "id": "wildcard_politei",
    "label": "form wildcard: πολιτει*",
    "arguments": {
      "query": "πολιτει*",
      "language": "greek",
      "query_format": "unicode",
      "search_kind": "form",
      "preserve_operators": true,
      "text_group": "urn:cts:greekLit:tlg0018",
      "result_format": "instances"
    }
  },
  {
    "id": "surface_variants",
    "label": "explicit surface variants",
    "arguments": {
      "query": "πολιτεία | πολιτείας | πολιτείᾳ | πολιτείαν | πολιτεῖαι | πολιτειῶν | πολιτείαις",
      "language": "greek",
      "query_format": "unicode",
      "search_kind": "form",
      "preserve_operators": true,
      "text_group": "urn:cts:greekLit:tlg0018",
  

## 11 - Run bounded, paged searches <a class="anchor" id="search-run"></a>
##### [Back to ToC](#TOC)

The collector repeats every argument exactly and changes only `page_num`. It stops when the result page is empty, the reported last page is reached, or the research limit is reached.

The summary records both upstream totals and how many pages/results this notebook actually observed. A total count is not equivalent to collected evidence.

In [13]:
async def collect_library_search_pages(client, strategy, max_pages):
    pages = []
    for page_num in range(1, max_pages + 1):
        data = await call_json(
            client,
            "search_perseus",
            {**strategy["arguments"], "page_num": page_num},
        )
        pages.append(data)
        results = data.get("results") or []
        page = data.get("page") or {}
        if not results:
            break
        current_page = page.get("number")
        num_pages = page.get("num_pages")
        if isinstance(current_page, int) and isinstance(num_pages, int) and current_page >= num_pages:
            break
    return pages


search_runs = {}
async with Client(mcp) as client:
    for strategy in SEARCH_PLAN:
        search_runs[strategy["id"]] = await collect_library_search_pages(
            client,
            strategy,
            MAX_PAGES_PER_STRATEGY,
        )

search_run_summary = {}
for strategy in SEARCH_PLAN:
    pages = search_runs[strategy["id"]]
    first = pages[0] if pages else {}
    search_run_summary[strategy["id"]] = {
        "label": strategy["label"],
        "query": strategy["arguments"]["query"],
        "upstream_total_count": first.get("total_count"),
        "upstream_num_pages": (first.get("page") or {}).get("num_pages"),
        "pages_collected": len(pages),
        "results_collected_before_deduplication": sum(
            len(page.get("results") or []) for page in pages
        ),
    }

print(json.dumps(search_run_summary, ensure_ascii=False, indent=2))

{
  "lemma_politeia": {
    "label": "lemma: πολιτεία",
    "query": "πολιτεία",
    "upstream_total_count": 0,
    "upstream_num_pages": 0,
    "pages_collected": 1,
    "results_collected_before_deduplication": 0
  },
  "wildcard_politei": {
    "label": "form wildcard: πολιτει*",
    "query": "πολιτει*",
    "upstream_total_count": 103,
    "upstream_num_pages": 11,
    "pages_collected": 3,
    "results_collected_before_deduplication": 30
  },
  "surface_variants": {
    "label": "explicit surface variants",
    "query": "πολιτεία | πολιτείας | πολιτείᾳ | πολιτείαν | πολιτεῖαι | πολιτειῶν | πολιτείαις",
    "upstream_total_count": 103,
    "upstream_num_pages": 11,
    "pages_collected": 3,
    "results_collected_before_deduplication": 30
  }
}


## 12 - Normalize and merge search candidates <a class="anchor" id="normalize-candidates"></a>
##### [Back to ToC](#TOC)

The same passage can appear in multiple strategies or pages. Candidate normalization keeps:

- passage, text/edition, and work URNs;
- citation and labels;
- first clean snippet;
- every search strategy and page that found the passage.

Deduplication is by complete passage URN. Search-source overlap is evidence about retrieval sensitivity and should not be discarded.

In [14]:
def candidate_from_result(result, strategy, page_number):
    passage = result.get("passage") or result
    text = passage.get("text") or result.get("text") or {}
    passage_urn = passage.get("urn") or result.get("urn")
    text_urn = text.get("urn")
    if not text_urn and passage_urn:
        text_urn = passage_urn.rpartition(":")[0]
    work_urn = text_urn.rsplit(".", 1)[0] if text_urn and "." in text_urn else None

    labels = [
        ancestor.get("label")
        for ancestor in text.get("ancestors", []) or []
        if ancestor.get("label")
    ]
    if text.get("label"):
        labels.append(text["label"])
    labels = unique_nonempty(labels)

    return {
        "urn": passage_urn,
        "citation": passage.get("citation") or result.get("citation"),
        "text_urn": text_urn,
        "work_urn": work_urn,
        "labels": labels,
        "work_label": labels[-1] if labels else work_urn,
        "snippet": clean_snippet(result.get("content") or []),
        "sources": [strategy["id"]],
        "source_labels": [strategy["label"]],
        "source_pages": [f"{strategy['id']}:{page_number}"],
    }


candidate_by_urn = {}
for strategy in SEARCH_PLAN:
    for page_index, page in enumerate(search_runs[strategy["id"]], start=1):
        page_number = (page.get("page") or {}).get("number") or page_index
        for result in page.get("results") or []:
            candidate = candidate_from_result(result, strategy, page_number)
            urn = candidate["urn"]
            if not urn:
                continue
            if urn not in candidate_by_urn:
                candidate_by_urn[urn] = candidate
                continue
            existing = candidate_by_urn[urn]
            existing["sources"] = unique_nonempty(existing["sources"] + candidate["sources"])
            existing["source_labels"] = unique_nonempty(
                existing["source_labels"] + candidate["source_labels"]
            )
            existing["source_pages"] = unique_nonempty(
                existing["source_pages"] + candidate["source_pages"]
            )
            if not existing["snippet"] and candidate["snippet"]:
                existing["snippet"] = candidate["snippet"]

candidates = list(candidate_by_urn.values())
scope_violations = [
    candidate["urn"] for candidate in candidates
    if not candidate["urn"].startswith(PHILO_TEXTGROUP + ".")
]
assert not scope_violations, f"Search returned out-of-scope URNs: {scope_violations[:5]}"

print(f"Unique passage candidates: {len(candidates)}")
print(json.dumps(candidates[:3], ensure_ascii=False, indent=2))

Unique passage candidates: 51
[
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg020.1st1K-grc1:242",
    "citation": null,
    "text_urn": "urn:cts:greekLit:tlg0018.tlg020.1st1K-grc1",
    "work_urn": "urn:cts:greekLit:tlg0018.tlg020",
    "labels": [
      "Philo Judaeus",
      "De Abrahamo"
    ],
    "work_label": "De Abrahamo",
    "snippet": "οἷς ἅπασιν ἐφεδρεύων ὁ ἀστεῖος, ἐπειδὴ κατεῖδε τὰ σύμμαχα καὶ φίλα πρὸ μικροῦ νοσοῦντα καὶ πόλεμον ἀντ’ εἰρήνης ταῖς ἐννέα βασιλείαις γενόμενον, πρὸς τὰς πέντε τῶν τεττάρων περὶ κράτους ἀρχῆς ἁμιλλωμένων, ἐξαπιναίως καιροφυλακήσας ἐπιτίθεται, φιλοτιμούμενος δημοκρατίαν, τὴν ἀρίστην τῶν πολιτειῶν, ἀντὶ τυραννίδων καὶ δυναστειῶν ἐν τῇ ψυχῇ καταστήσασθαι καὶ τὸ ἔννομον καὶ τὸ δίκαιον ἀντὶ παρανομίας καὶ ἀδικίας, αἳ τέως ἐπεκράτουν.",
    "sources": [
      "wildcard_politei",
      "surface_variants"
    ],
    "source_labels": [
      "form wildcard: πολιτει*",
      "explicit surface variants"
    ],
    "source_pages": [
      "wildcard_politei:

## 13 - Inspect source overlap and work distribution <a class="anchor" id="coverage"></a>
##### [Back to ToC](#TOC)

Before sampling, inspect what the search stage produced:

- how many unique passages each strategy contributed;
- how often multiple strategies converged on the same passage;
- which works dominate the collected pages.

This prevents an accidental statement such as “Philo uses the term mainly in *De Josepho*” when the apparent dominance merely reflects ranking and page limits.

In [15]:
strategy_candidate_counts = Counter(
    source for candidate in candidates for source in candidate["sources"]
)
strategy_overlap_counts = Counter(len(candidate["sources"]) for candidate in candidates)
work_candidate_counts = Counter(
    candidate["work_label"] or candidate["work_urn"] or "[unknown work]"
    for candidate in candidates
)

coverage_report = {
    "unique_candidates": len(candidates),
    "strategy_candidate_counts": dict(strategy_candidate_counts),
    "number_of_strategies_per_candidate": dict(sorted(strategy_overlap_counts.items())),
    "work_candidate_counts": dict(work_candidate_counts.most_common()),
}
print(json.dumps(coverage_report, ensure_ascii=False, indent=2))

{
  "unique_candidates": 51,
  "strategy_candidate_counts": {
    "wildcard_politei": 30,
    "surface_variants": 30
  },
  "number_of_strategies_per_candidate": {
    "1": 42,
    "2": 9
  },
  "work_candidate_counts": {
    "De Specialibus Legibus (lib. i‑iv)": 10,
    "De Josepho": 8,
    "De Vita Mosis (Lib. I-II)": 4,
    "Legatio Ad Gaium": 4,
    "De Abrahamo": 3,
    "De Decalogo": 3,
    "Quod Omnis Probus Liber Sit": 3,
    "De Virtutibus": 2,
    "De Sacrificiis Abelis Et Caini": 2,
    "De Gigantibus": 2,
    "De Ebrietate": 2,
    "De Confusione Linguarum": 2,
    "De Somniis (lib. i-ii)": 2,
    "De Plantatione": 1,
    "Quis Rerum Divinarum Heres Sit": 1,
    "De Fuga Et Inventione": 1,
    "De Opificio Mundi": 1
  }
}


## 14 - Define a bounded stratified sampling policy <a class="anchor" id="sampling-policy"></a>
##### [Back to ToC](#TOC)

Taking the first ten ranked passages can overrepresent one work. This notebook uses a deterministic round-robin sample by work:

1. group candidates by work in first-seen order;
2. take the first candidate from each work;
3. take a second candidate from each work, then a third;
4. stop at the candidate-fetch limit;
5. retrieve text and retain only passages where the folded target stem is present;
6. stop the final evidence set at its own smaller limit.

This policy increases work coverage, but it is still purposive and ranking-dependent. It is not random sampling, proportional sampling, or exhaustive corpus analysis.

In [16]:
MAX_CANDIDATES_TO_FETCH = 18
MAX_EVIDENCE_PASSAGES = 12
MAX_PER_WORK_IN_FETCH_SET = 3
MAX_CONTEXT_CHARACTERS = 3000


def stratified_round_robin(records, max_records, max_per_work):
    grouped = defaultdict(list)
    work_order = []
    for record in records:
        key = record.get("work_urn") or "[unknown work]"
        if key not in grouped:
            work_order.append(key)
        grouped[key].append(record)

    selected = []
    for depth in range(max_per_work):
        for work in work_order:
            if depth < len(grouped[work]):
                selected.append(grouped[work][depth])
                if len(selected) >= max_records:
                    return selected
    return selected


print(
    json.dumps(
        {
            "max_candidates_to_fetch": MAX_CANDIDATES_TO_FETCH,
            "max_final_evidence_passages": MAX_EVIDENCE_PASSAGES,
            "max_candidates_per_work": MAX_PER_WORK_IN_FETCH_SET,
            "max_context_characters_per_passage": MAX_CONTEXT_CHARACTERS,
        },
        indent=2,
    )
)

{
  "max_candidates_to_fetch": 18,
  "max_final_evidence_passages": 12,
  "max_candidates_per_work": 3,
  "max_context_characters_per_passage": 3000
}


## 15 - Select candidate passages across works <a class="anchor" id="sample-candidates"></a>
##### [Back to ToC](#TOC)

This is the pre-retrieval sample. Some records may later be excluded because the passage cannot be fetched or the target stem does not appear in the retrieved text.

In [17]:
selected_candidates = stratified_round_robin(
    candidates,
    max_records=MAX_CANDIDATES_TO_FETCH,
    max_per_work=MAX_PER_WORK_IN_FETCH_SET,
)

selection_report = [
    {
        "urn": candidate["urn"],
        "work": candidate["work_label"],
        "sources": candidate["sources"],
        "snippet_preview": candidate["snippet"][:220],
    }
    for candidate in selected_candidates
]
print(json.dumps(selection_report, ensure_ascii=False, indent=2))

[
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg020.1st1K-grc1:242",
    "work": "De Abrahamo",
    "sources": [
      "wildcard_politei",
      "surface_variants"
    ],
    "snippet_preview": "οἷς ἅπασιν ἐφεδρεύων ὁ ἀστεῖος, ἐπειδὴ κατεῖδε τὰ σύμμαχα καὶ φίλα πρὸ μικροῦ νοσοῦντα καὶ πόλεμον ἀντ’ εἰρήνης ταῖς ἐννέα βασιλείαις γενόμενον, πρὸς τὰς πέντε τῶν τεττάρων περὶ κράτους ἀρχῆς ἁμιλλωμένων, ἐξαπιναίως καιρ"
  },
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg021.1st1K-grc1:29",
    "work": "De Josepho",
    "sources": [
      "wildcard_politei",
      "surface_variants"
    ],
    "snippet_preview": "ἡ μὲν γὰρ μεγαλόπολις ὅδε ὁ κόσμος ἐστὶ καὶ μιᾷ χρῆται πολιτείᾳ καὶ νόμῳ ἑνί· λόγος δέ ἐστι φύσεως προστακτικὸς μὲν ὧν πρακτέον, ἀπαγορευτικὸς δὲ ὧν οὐ ποιητέον· αἱ δὲ κατὰ τόπους αὗται πόλεις ἀπερίγραφοί τέ εἰσιν ἀριθμῷ"
  },
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg022.1st1K-grc1:1.253",
    "work": "De Vita Mosis (Lib. I-II)",
    "sources": [
      "wildcard_politei"
    ],
    "sn

## 16 - Retrieve passage text and center context on the target stem <a class="anchor" id="fetch-contexts"></a>
##### [Back to ToC](#TOC)

Search snippets are not enough for interpretation. Each selected candidate is fetched through `get_scaife_passage_text`.

The context helper searches an accent-folded copy for `πολιτει` and maps the match back to the original accented string. This is especially useful when a passage contains a long critical apparatus before or after the lexical occurrence.

The notebook records failures rather than replacing them with fabricated text.

In [18]:
retrieved_candidates = []
async with Client(mcp) as client:
    for candidate in selected_candidates:
        record = {**candidate}
        try:
            passage_text = await call_text(
                client,
                "get_scaife_passage_text",
                {"urn": candidate["urn"]},
            )
            context, target_present, context_clipped = centered_context(
                passage_text,
                max_chars=MAX_CONTEXT_CHARACTERS,
            )
            record.update(
                {
                    "retrieval_status": "ok",
                    "passage_text_characters": len(passage_text),
                    "target_stem_present": target_present,
                    "context_clipped": context_clipped,
                    "context": context,
                }
            )
        except Exception as exc:
            record.update(
                {
                    "retrieval_status": "error",
                    "retrieval_error": f"{type(exc).__name__}: {exc}",
                    "target_stem_present": False,
                    "context_clipped": False,
                    "context": None,
                }
            )
        retrieved_candidates.append(record)

print(
    json.dumps(
        [
            {
                "urn": row["urn"],
                "work": row["work_label"],
                "status": row["retrieval_status"],
                "target_present": row["target_stem_present"],
                "context_clipped": row["context_clipped"],
                "text_characters": row.get("passage_text_characters"),
                "error": row.get("retrieval_error"),
            }
            for row in retrieved_candidates
        ],
        ensure_ascii=False,
        indent=2,
    )
)

[
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg020.1st1K-grc1:242",
    "work": "De Abrahamo",
    "status": "ok",
    "target_present": true,
    "context_clipped": false,
    "text_characters": 435,
    "error": null
  },
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg021.1st1K-grc1:29",
    "work": "De Josepho",
    "status": "ok",
    "target_present": true,
    "context_clipped": false,
    "text_characters": 357,
    "error": null
  },
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg022.1st1K-grc1:1.253",
    "work": "De Vita Mosis (Lib. I-II)",
    "status": "ok",
    "target_present": true,
    "context_clipped": false,
    "text_characters": 1285,
    "error": null
  },
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg023.1st1K-grc1:155",
    "work": "De Decalogo",
    "status": "ok",
    "target_present": true,
    "context_clipped": false,
    "text_characters": 310,
    "error": null
  },
  {
    "urn": "urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:3.51",
    "work": "De Specialibus Legib

## 17 - Audit retrieval and lexical relevance <a class="anchor" id="quality-audit"></a>
##### [Back to ToC](#TOC)

A passage enters the final evidence set only if:

- retrieval succeeded;
- a nonempty context was returned;
- the accent-folded target stem `πολιτει` is present.

This lexical check is intentionally modest. It confirms the context contains a form beginning with the target stem; it does not establish morphology, lemma identity, or sense.

In [19]:
eligible_evidence = [
    row for row in retrieved_candidates
    if row["retrieval_status"] == "ok"
    and row.get("context")
    and row["target_stem_present"]
]
evidence_rows = eligible_evidence[:MAX_EVIDENCE_PASSAGES]
excluded_candidates = []
for row in retrieved_candidates:
    if row in evidence_rows:
        continue
    if row["retrieval_status"] != "ok":
        reason = "retrieval failed"
    elif not row["target_stem_present"]:
        reason = "target stem absent from retrieved passage text"
    else:
        reason = "final evidence passage limit reached"
    excluded_candidates.append(
        {
            "urn": row["urn"],
            "work": row["work_label"],
            "status": row["retrieval_status"],
            "target_stem_present": row["target_stem_present"],
            "reason": reason,
            "error": row.get("retrieval_error"),
        }
    )

quality_report = {
    "candidates_selected_for_fetch": len(selected_candidates),
    "retrieval_successes": sum(row["retrieval_status"] == "ok" for row in retrieved_candidates),
    "target_relevant_successes": len(eligible_evidence),
    "final_evidence_passages": len(evidence_rows),
    "excluded": excluded_candidates,
}
print(json.dumps(quality_report, ensure_ascii=False, indent=2))

if not evidence_rows:
    raise RuntimeError("No retrievable target-relevant evidence remains after quality checks.")

{
  "candidates_selected_for_fetch": 18,
  "retrieval_successes": 18,
  "target_relevant_successes": 18,
  "final_evidence_passages": 12,
  "excluded": [
    {
      "urn": "urn:cts:greekLit:tlg0018.tlg013.1st1K-grc1:108",
      "work": "De Confusione Linguarum",
      "status": "ok",
      "target_stem_present": true,
      "reason": "final evidence passage limit reached",
      "error": null
    },
    {
      "urn": "urn:cts:greekLit:tlg0018.tlg015.1st1K-grc1:169",
      "work": "Quis Rerum Divinarum Heres Sit",
      "status": "ok",
      "target_stem_present": true,
      "reason": "final evidence passage limit reached",
      "error": null
    },
    {
      "urn": "urn:cts:greekLit:tlg0018.tlg017.1st1K-grc1:35",
      "work": "De Fuga Et Inventione",
      "status": "ok",
      "target_stem_present": true,
      "reason": "final evidence passage limit reached",
      "error": null
    },
    {
      "urn": "urn:cts:greekLit:tlg0018.tlg019.1st1K-grc1:1.220",
      "work": "De Som

## 18 - Build a descriptive baseline before interpretation <a class="anchor" id="baseline"></a>
##### [Back to ToC](#TOC)

Before asking an LLM for semantic categories, compute simple observations that require no interpretive model:

- evidence passages by work;
- retrieval strategies represented in the final evidence;
- frequent non-stopword neighbors within five Greek tokens of a target-stem token.

The neighbor list is not lemmatized, morphologically parsed, or statistically normalized. It is an exploratory prompt for human follow-up, not a collocation study.

In [20]:
GREEK_STOPWORDS = {
    "και", "δε", "τε", "των", "την", "της", "τω", "τα", "το", "του",
    "εν", "εις", "εκ", "επι", "προς", "μεν", "ουν", "γαρ", "ως", "ο", "η",
    "οι", "αι", "τον", "τους", "τας", "αυτου", "αυτων", "ου", "μη",
}


def folded_greek_tokens(text):
    return [fold_greek(token) for token in GREEK_TOKEN_RE.findall(text or "")]


neighbor_counts = Counter()
for row in evidence_rows:
    tokens = folded_greek_tokens(row["context"])
    for index, token in enumerate(tokens):
        if TARGET_FOLDED_STEM not in token:
            continue
        for neighbor in tokens[max(0, index - 5) : index + 6]:
            if neighbor == token or neighbor in GREEK_STOPWORDS or len(neighbor) < 3:
                continue
            neighbor_counts[neighbor] += 1

baseline_report = {
    "evidence_by_work": dict(
        Counter(row["work_label"] or row["work_urn"] for row in evidence_rows).most_common()
    ),
    "evidence_by_search_strategy": dict(
        Counter(source for row in evidence_rows for source in row["sources"]).most_common()
    ),
    "frequent_local_neighbors_unlemmatized": neighbor_counts.most_common(20),
}
print(json.dumps(baseline_report, ensure_ascii=False, indent=2))

{
  "evidence_by_work": {
    "De Abrahamo": 1,
    "De Josepho": 1,
    "De Vita Mosis (Lib. I-II)": 1,
    "De Decalogo": 1,
    "De Specialibus Legibus (lib. i‑iv)": 1,
    "De Virtutibus": 1,
    "Quod Omnis Probus Liber Sit": 1,
    "Legatio Ad Gaium": 1,
    "De Sacrificiis Abelis Et Caini": 1,
    "De Gigantibus": 1,
    "De Plantatione": 1,
    "De Ebrietate": 1
  },
  "evidence_by_search_strategy": {
    "wildcard_politei": 12,
    "surface_variants": 5
  },
  "frequent_local_neighbors_unlemmatized": [
    [
      "προσ",
      4
    ],
    [
      "σωφροσυνησ",
      3
    ],
    [
      "αυτουσ",
      2
    ],
    [
      "μωυσην",
      2
    ],
    [
      "ιερον",
      2
    ],
    [
      "λογον",
      2
    ],
    [
      "φιλοτιμουμενοσ",
      1
    ],
    [
      "δημοκρατιαν",
      1
    ],
    [
      "αριστην",
      1
    ],
    [
      "αντι",
      1
    ],
    [
      "τυραννιδων",
      1
    ],
    [
      "δυναστειων",
      1
    ],
    [
      "κοσμοσ

## 19 - Assemble the research evidence packet <a class="anchor" id="evidence-packet"></a>
##### [Back to ToC](#TOC)

The packet records scope, methodology, search coverage, sampling limits, descriptive observations, evidence passages, and known limitations.

Only centered contexts—not unrestricted full passages—are included in the model packet. This bounds context size while preserving the exact passage URN and retrieval provenance.

In [21]:
packet_evidence = [
    {
        "urn": row["urn"],
        "citation": row["citation"],
        "text_urn": row["text_urn"],
        "work_urn": row["work_urn"],
        "work_label": row["work_label"],
        "labels": row["labels"],
        "found_by": row["source_labels"],
        "source_pages": row["source_pages"],
        "search_snippet": row["snippet"],
        "context": row["context"],
        "context_clipped": row["context_clipped"],
        "target_stem_present": row["target_stem_present"],
    }
    for row in evidence_rows
]

evidence_packet = {
    "research_question": (
        "What semantic and metaphorical uses of the πολιτεία lexical family are visible "
        "in this bounded sample of Scaife-indexed Greek passages attributed to Philo?"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": {
        "textgroup_urn": PHILO_TEXTGROUP,
        "label": philo_label,
        "service": "Scaife library/search",
    },
    "methodology": {
        "search_plan": SEARCH_PLAN,
        "max_pages_per_strategy": MAX_PAGES_PER_STRATEGY,
        "search_run_summary": search_run_summary,
        "unique_candidate_count": len(candidates),
        "sampling": {
            "method": "deterministic round-robin by first-seen work",
            "max_candidates_fetched": MAX_CANDIDATES_TO_FETCH,
            "max_per_work_in_fetch_set": MAX_PER_WORK_IN_FETCH_SET,
            "max_final_evidence_passages": MAX_EVIDENCE_PASSAGES,
            "max_context_characters": MAX_CONTEXT_CHARACTERS,
        },
        "quality_filter": (
            "retrieval succeeded and accent-folded context contains the stem πολιτει"
        ),
    },
    "descriptive_baseline": baseline_report,
    "evidence": packet_evidence,
    "excluded_candidate_summary": excluded_candidates,
    "limitations": [
        "Searches were bounded to a maximum number of pages per strategy.",
        "The sample is stratified by work but remains dependent on Scaife ranking.",
        "Wildcard and OR-style semantics are controlled by the upstream search service.",
        "Lemma search coverage depends on upstream morphological indexing.",
        "Passage text may contain critical apparatus or editorial material.",
        "The packet contains Greek text but no supplied translation.",
        "Citation membership does not prove that an interpretive claim is entailed by a passage.",
    ],
}

evidence_packet_json = json.dumps(evidence_packet, ensure_ascii=False, indent=2)
print(f"Evidence packet characters: {len(evidence_packet_json):,}")
print(evidence_packet_json[:8000])

Evidence packet characters: 31,903
{
  "research_question": "What semantic and metaphorical uses of the πολιτεία lexical family are visible in this bounded sample of Scaife-indexed Greek passages attributed to Philo?",
  "created_at_utc": "2026-06-18T19:57:02.652756+00:00",
  "scope": {
    "textgroup_urn": "urn:cts:greekLit:tlg0018",
    "label": "Philo Judaeus",
    "service": "Scaife library/search"
  },
  "methodology": {
    "search_plan": [
      {
        "id": "lemma_politeia",
        "label": "lemma: πολιτεία",
        "arguments": {
          "query": "πολιτεία",
          "language": "greek",
          "query_format": "unicode",
          "search_kind": "lemma",
          "text_group": "urn:cts:greekLit:tlg0018",
          "result_format": "instances"
        }
      },
      {
        "id": "wildcard_politei",
        "label": "form wildcard: πολιτει*",
        "arguments": {
          "query": "πολιτει*",
          "language": "greek",
          "query_format": "unicode",

## 20 - Validate packet provenance and size <a class="anchor" id="packet-validation"></a>
##### [Back to ToC](#TOC)

The deterministic packet audit checks:

- unique evidence URNs;
- Philo textgroup scope;
- target-stem presence;
- nonempty contexts;
- known search-source IDs;
- a configurable character budget.

Character count is only a rough proxy for tokens. Consult the selected model's context length and actual API usage.

In [22]:
MAX_EVIDENCE_PACKET_CHARACTERS = 70_000


def validate_evidence_packet(packet):
    evidence = packet.get("evidence") or []
    urns = [row.get("urn") for row in evidence]
    known_sources = {strategy["label"] for strategy in SEARCH_PLAN}
    return {
        "evidence_count": len(evidence),
        "duplicate_urns": sorted(
            urn for urn, count in Counter(urns).items() if urn and count > 1
        ),
        "missing_urns": [index for index, urn in enumerate(urns) if not urn],
        "out_of_scope_urns": sorted(
            urn for urn in urns if urn and not urn.startswith(PHILO_TEXTGROUP + ".")
        ),
        "empty_context_urns": sorted(
            row["urn"] for row in evidence if not row.get("context")
        ),
        "target_missing_urns": sorted(
            row["urn"] for row in evidence if not row.get("target_stem_present")
        ),
        "unknown_source_labels": sorted(
            {
                source
                for row in evidence
                for source in row.get("found_by", [])
                if source not in known_sources
            }
        ),
        "packet_characters": len(json.dumps(packet, ensure_ascii=False)),
        "over_character_budget": len(json.dumps(packet, ensure_ascii=False)) > MAX_EVIDENCE_PACKET_CHARACTERS,
    }


packet_validation = validate_evidence_packet(evidence_packet)
print(json.dumps(packet_validation, ensure_ascii=False, indent=2))

critical_packet_issues = {
    key: value
    for key, value in packet_validation.items()
    if key not in {"evidence_count", "packet_characters"} and value
}
assert not critical_packet_issues, f"Evidence packet validation failed: {critical_packet_issues}"

{
  "evidence_count": 12,
  "duplicate_urns": [],
  "missing_urns": [],
  "out_of_scope_urns": [],
  "empty_context_urns": [],
  "target_missing_urns": [],
  "unknown_source_labels": [],
  "packet_characters": 28493,
  "over_character_budget": false
}


## 21 - Define the structured analysis contract <a class="anchor" id="analysis-contract"></a>
##### [Back to ToC](#TOC)

The model must return one JSON object with this conceptual shape:

```json
{
  "thesis": "A sample-bounded thesis",
  "categories": [
    {
      "label": "Short category label",
      "claim": "What the supplied passages show",
      "evidence_urns": ["exact packet URN"],
      "confidence": "high | medium | low",
      "qualification": "Ambiguity or limit"
    }
  ],
  "limitations": ["..."],
  "follow_up_searches": [
    {"query": "...", "reason": "..."}
  ]
}
```

Contract rules:

- use only the evidence packet;
- make claims about the bounded sample, not all of Philo;
- cite exact supplied URNs in every category;
- do not invent translations, passages, work titles, or lexical statistics;
- do not create a category merely to reach a requested number;
- distinguish direct lexical use, metaphorical extension, and uncertain cases;
- return JSON only.

In [23]:
ANALYSIS_SYSTEM_PROMPT = """You are a cautious scholar of Hellenistic Jewish Greek.
Use only the evidence packet supplied by the user. Treat it as a bounded sample,
not an exhaustive corpus. Do not use outside passages, translations, statistics,
or historical claims. Every semantic category must cite one or more exact URNs
from the packet. Do not infer that citation membership automatically proves a
claim: qualify ambiguity explicitly. Return only valid JSON."""


def build_analysis_user_prompt(packet):
    return f"""Analyze the uses of the πολιτεία lexical family visible in this evidence packet.

Return exactly one JSON object with keys:
- thesis: string, explicitly limited to this sample;
- categories: list of 2-5 objects, each with label, claim, evidence_urns,
  confidence (high/medium/low), and qualification;
- limitations: list of strings;
- follow_up_searches: list of objects with query and reason.

Do not supply a translation unless the packet supplies one (it does not).
Use exact complete URNs copied from packet.evidence[].urn.
If fewer than two categories are defensible, return fewer and explain why.

Evidence packet:
{json.dumps(packet, ensure_ascii=False, indent=2)}"""


analysis_user_prompt = build_analysis_user_prompt(evidence_packet)
print(f"Analysis prompt characters: {len(analysis_user_prompt):,}")

Analysis prompt characters: 32,525


## 22 - Build a defensive OpenRouter JSON helper <a class="anchor" id="openrouter-client"></a>
##### [Back to ToC](#TOC)

The helper uses an asynchronous HTTP client, reports HTTP errors without exposing request headers, records model/usage metadata, and parses either plain or fenced JSON. It reserves enough completion budget for the structured answer and requests low reasoning effort when the selected model supports reasoning.

If the selected model advertises `response_format`, the request asks for a JSON object. The Free Models Router is treated as supporting this request feature so OpenRouter can select a compatible free model. The prompt still requires JSON because provider behavior can vary.

In [24]:
async def openrouter_json_completion(
    messages,
    *,
    model=OPENROUTER_MODEL,
    temperature=0.1,
    max_tokens=6000,
    timeout_seconds=120.0,
):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    supported = set((selected_model_info or {}).get("supported_parameters") or [])
    if "response_format" in supported:
        payload["response_format"] = {"type": "json_object"}
    if "reasoning" in supported:
        payload["reasoning"] = {"effort": "low", "exclude": True}

    async with httpx.AsyncClient(timeout=timeout_seconds) as http:
        response = await http.post(
            OPENROUTER_CHAT_URL,
            headers=openrouter_headers(),
            json=payload,
        )

    try:
        response.raise_for_status()
    except httpx.HTTPStatusError as exc:
        raise RuntimeError(
            f"OpenRouter error {response.status_code}: {response.text[:2000]}"
        ) from exc

    try:
        data = response.json()
    except ValueError as exc:
        raise RuntimeError(
            f"OpenRouter returned non-JSON HTTP content: {response.text[:1000]}"
        ) from exc

    if not data.get("choices") or not data["choices"][0].get("message"):
        raise RuntimeError(f"OpenRouter returned no assistant message: {json.dumps(data)[:2000]}")

    finish_reason = data["choices"][0].get("finish_reason")
    raw_content = data["choices"][0]["message"].get("content") or ""
    try:
        parsed = extract_json_object(raw_content)
        parse_error = None
    except Exception as exc:
        parsed = None
        if finish_reason == "length":
            parse_error = (
                f"TruncatedResponseError: OpenRouter reached max_tokens={max_tokens} "
                "before completing the JSON response."
            )
        else:
            parse_error = f"{type(exc).__name__}: {exc}"

    return {
        "response_id": data.get("id"),
        "requested_model": model,
        "resolved_model": data.get("model"),
        "finish_reason": finish_reason,
        "usage": data.get("usage") or {},
        "raw_content": raw_content,
        "parsed": parsed,
        "parse_error": parse_error,
    }

## 23 - Optionally request a cited synthesis <a class="anchor" id="synthesis"></a>
##### [Back to ToC](#TOC)

This call is disabled by default. Enabling it sends the evidence packet—including retrieved Greek contexts—to OpenRouter and may incur charges.

The raw response is retained alongside the parsed JSON and usage metadata.

In [30]:
RUN_LLM_SYNTHESIS = False
analysis_run = None
analysis_payload = None

if RUN_LLM_SYNTHESIS:
    analysis_run = await openrouter_json_completion(
        [
            {"role": "system", "content": ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": analysis_user_prompt},
        ],
        temperature=0.1,
    )
    analysis_payload = analysis_run["parsed"]
    print(
        json.dumps(
            {
                "response_id": analysis_run["response_id"],
                "resolved_model": analysis_run["resolved_model"],
                "finish_reason": analysis_run["finish_reason"],
                "usage": analysis_run["usage"],
                "parse_error": analysis_run["parse_error"],
            },
            ensure_ascii=False,
            indent=2,
        )
    )
    if analysis_payload is None:
        print("\nRaw response (clipped):")
        print(analysis_run["raw_content"][:5000])
else:
    print("Skipping OpenRouter synthesis. Set RUN_LLM_SYNTHESIS = True to run it.")

Skipping OpenRouter synthesis. Set RUN_LLM_SYNTHESIS = True to run it.


## 24 - Audit model citations deterministically <a class="anchor" id="citation-audit"></a>
##### [Back to ToC](#TOC)

The audit verifies structure and provenance:

- required top-level fields;
- categories are objects with evidence lists;
- confidence labels are recognized;
- every category has at least one URN;
- every cited URN belongs to the packet;
- URNs appearing anywhere in the raw response are also checked.

This does **not** prove that a claim is entailed by the cited passage. Semantic support still requires reading the context and, optionally, running the skeptical review.

In [31]:
ALLOWED_EVIDENCE_URNS = {row["urn"] for row in evidence_packet["evidence"]}


def audit_analysis_payload(payload, allowed_urns, raw_content=""):
    if not isinstance(payload, dict):
        return {"valid": False, "errors": ["Analysis payload is not a JSON object."]}

    errors = []
    warnings = []
    required_keys = {"thesis", "categories", "limitations", "follow_up_searches"}
    missing_keys = sorted(required_keys - set(payload))
    if missing_keys:
        errors.append(f"Missing top-level keys: {missing_keys}")
    if not isinstance(payload.get("thesis"), str) or not payload.get("thesis", "").strip():
        errors.append("thesis must be a nonempty string")
    if not isinstance(payload.get("limitations"), list):
        errors.append("limitations must be a list")
    if not isinstance(payload.get("follow_up_searches"), list):
        errors.append("follow_up_searches must be a list")

    categories = payload.get("categories")
    if not isinstance(categories, list):
        errors.append("categories must be a list")
        categories = []

    category_reports = []
    cited = set()
    for index, category in enumerate(categories):
        if not isinstance(category, dict):
            errors.append(f"category {index} is not an object")
            continue
        for field in ["label", "claim", "qualification"]:
            if not isinstance(category.get(field), str) or not category.get(field, "").strip():
                errors.append(f"category {index} has no nonempty {field}")
        urns = category.get("evidence_urns")
        if not isinstance(urns, list) or not urns:
            errors.append(f"category {index} has no evidence_urns list")
            urns = []
        unknown = sorted(set(urns) - allowed_urns)
        if unknown:
            errors.append(f"category {index} cites unknown URNs: {unknown}")
        cited.update(urns)
        confidence = category.get("confidence")
        if confidence not in {"high", "medium", "low"}:
            warnings.append(f"category {index} has nonstandard confidence: {confidence!r}")
        category_reports.append(
            {
                "index": index,
                "label": category.get("label"),
                "cited_urns": urns,
                "unknown_urns": unknown,
            }
        )

    raw_urns = set(extract_cts_urns(raw_content))
    raw_unknown = sorted(raw_urns - allowed_urns)
    if raw_unknown:
        errors.append(f"Raw response contains unknown URNs: {raw_unknown}")

    return {
        "valid": not errors,
        "errors": errors,
        "warnings": warnings,
        "category_reports": category_reports,
        "cited_packet_urns": sorted(cited & allowed_urns),
        "uncited_packet_urns": sorted(allowed_urns - cited),
        "raw_response_urns": sorted(raw_urns),
    }


analysis_audit = None
if analysis_run is not None:
    analysis_audit = audit_analysis_payload(
        analysis_payload,
        ALLOWED_EVIDENCE_URNS,
        analysis_run["raw_content"],
    )
    print(json.dumps(analysis_audit, ensure_ascii=False, indent=2))
else:
    print("No synthesis was run, so there is no model citation audit yet.")

No synthesis was run, so there is no model citation audit yet.


## 25 - Render the structured analysis <a class="anchor" id="render-analysis"></a>
##### [Back to ToC](#TOC)

Rendering occurs only after JSON parsing. If the citation audit fails, the notebook displays a warning before any polished prose.

The renderer does not silently remove unsupported categories; it keeps the generated structure inspectable.

In [32]:
def markdown_escape_cell(value):
    return str(value or "").replace("|", "\\|").replace("\n", " ")


def render_analysis_markdown(payload, title="Sample-bounded analysis"):
    lines = [f"## {title}", "", str(payload.get("thesis") or "[No thesis returned]"), ""]
    categories = payload.get("categories") or []
    if categories:
        lines.extend(
            [
                "| Category | Claim | Evidence URNs | Confidence | Qualification |",
                "|---|---|---|---|---|",
            ]
        )
        for category in categories:
            urns = "<br>".join(f"`{urn}`" for urn in category.get("evidence_urns", []))
            lines.append(
                "| "
                + " | ".join(
                    [
                        markdown_escape_cell(category.get("label")),
                        markdown_escape_cell(category.get("claim")),
                        urns,
                        markdown_escape_cell(category.get("confidence")),
                        markdown_escape_cell(category.get("qualification")),
                    ]
                )
                + " |"
            )

    lines.extend(["", "### Limitations", ""])
    lines.extend(f"- {item}" for item in payload.get("limitations") or ["[None returned]"])
    lines.extend(["", "### Follow-up searches", ""])
    for item in payload.get("follow_up_searches") or []:
        lines.append(f"- `{item.get('query', '')}` — {item.get('reason', '')}")
    return "\n".join(lines)


if analysis_payload is not None:
    if analysis_audit and analysis_audit["valid"]:
        display(Markdown(render_analysis_markdown(analysis_payload)))
    else:
        errors = (analysis_audit or {}).get("errors", ["No successful audit was produced."])
        display(Markdown("## ⚠ Citation/structure audit failed\n\n" + "\n".join(
            f"- {error}" for error in errors
        )))
        print(json.dumps(analysis_payload, ensure_ascii=False, indent=2))
else:
    print("No parsed synthesis is available to render.")

No parsed synthesis is available to render.


## 26 - Optionally run a skeptical review <a class="anchor" id="skeptical-review"></a>
##### [Back to ToC](#TOC)

The critic sees the same evidence packet plus the structured analysis. It is asked to identify:

- claims that exceed the cited context;
- conflated lexical senses;
- category boundaries that are too confident;
- missing qualifications;
- evidence needed to strengthen or falsify the analysis.

The critic is another model output, not an authority. Its citations are also checked against the packet.

In [33]:
RUN_CRITIC_PASS = False
critic_run = None
critic_payload = None
critic_citation_audit = None

if RUN_CRITIC_PASS:
    if analysis_payload is None:
        raise RuntimeError("Run and parse the synthesis before enabling the critic pass.")

    critic_prompt = f"""Review the proposed analysis against the evidence packet only.

Return one JSON object with:
- verdict: string;
- findings: list of objects with severity (high/medium/low), claim,
  assessment, evidence_urns, and recommendation;
- missing_evidence: list of strings;
- revision_priorities: list of strings.

Do not add outside passages or translations. Use only exact packet URNs.

Evidence packet:
{json.dumps(evidence_packet, ensure_ascii=False, indent=2)}

Analysis to review:
{json.dumps(analysis_payload, ensure_ascii=False, indent=2)}"""

    critic_run = await openrouter_json_completion(
        [
            {
                "role": "system",
                "content": (
                    "You are a skeptical philological reviewer. Use only supplied evidence, "
                    "separate provenance from entailment, and return JSON only."
                ),
            },
            {"role": "user", "content": critic_prompt},
        ],
        temperature=0.0,
    )
    critic_payload = critic_run["parsed"]
    critic_urns = set(extract_cts_urns(critic_payload or critic_run["raw_content"]))
    critic_citation_audit = {
        "cited_urns": sorted(critic_urns),
        "unknown_urns": sorted(critic_urns - ALLOWED_EVIDENCE_URNS),
        "valid": not bool(critic_urns - ALLOWED_EVIDENCE_URNS),
    }
    print(json.dumps(critic_citation_audit, ensure_ascii=False, indent=2))
    if critic_payload is not None:
        display(Markdown("## Skeptical review\n\n```json\n" + json.dumps(
            critic_payload, ensure_ascii=False, indent=2
        ) + "\n```"))
else:
    print("Skipping critic pass. Set RUN_CRITIC_PASS = True after synthesis to run it.")

Skipping critic pass. Set RUN_CRITIC_PASS = True after synthesis to run it.


## 27 - Optionally produce an evidence-constrained revision <a class="anchor" id="revision"></a>
##### [Back to ToC](#TOC)

A revision is a third, separate call. It receives the same evidence plus the original analysis and critique, and must return the original analysis schema.

Keeping revision separate prevents the first answer from being silently overwritten and allows before/after comparison.

In [34]:
RUN_REVISION_PASS = False
revision_run = None
revision_payload = None
revision_audit = None

if RUN_REVISION_PASS:
    if analysis_payload is None or critic_payload is None:
        raise RuntimeError("A parsed synthesis and parsed critique are required for revision.")

    revision_prompt = f"""Revise the analysis to address the skeptical review.
Use only the evidence packet. Return the same JSON schema as the original analysis:
thesis, categories, limitations, follow_up_searches. Preserve exact packet URNs,
remove unsupported claims, lower confidence where needed, and keep the thesis
explicitly sample-bounded.

Evidence packet:
{json.dumps(evidence_packet, ensure_ascii=False, indent=2)}

Original analysis:
{json.dumps(analysis_payload, ensure_ascii=False, indent=2)}

Critique:
{json.dumps(critic_payload, ensure_ascii=False, indent=2)}"""

    revision_run = await openrouter_json_completion(
        [
            {"role": "system", "content": ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": revision_prompt},
        ],
        temperature=0.0,
    )
    revision_payload = revision_run["parsed"]
    revision_audit = audit_analysis_payload(
        revision_payload,
        ALLOWED_EVIDENCE_URNS,
        revision_run["raw_content"],
    )
    print(json.dumps(revision_audit, ensure_ascii=False, indent=2))
    if revision_payload is not None and revision_audit["valid"]:
        display(Markdown(render_analysis_markdown(revision_payload, "Revised sample-bounded analysis")))
    elif revision_payload is not None:
        print("Revision failed citation/structure audit; raw parsed JSON follows:")
        print(json.dumps(revision_payload, ensure_ascii=False, indent=2))
else:
    print("Skipping revision. Enable it only after synthesis and critique.")

Skipping revision. Enable it only after synthesis and critique.


## 28 - Interpret the result cautiously <a class="anchor" id="interpretive-cautions"></a>
##### [Back to ToC](#TOC)

Several distinctions must survive the final prose:

### Search coverage versus lexical absence

A zero lemma count does not prove that the lemma is absent. It can expose a gap or mismatch in morphological indexing. Wildcard and explicit-form searches provide complementary evidence, not a perfect substitute.

### Ranked sample versus corpus distribution

The sample comes from a bounded number of ranked pages and a deterministic work-stratified policy. Work counts in the sample are not corpus frequencies.

### Stem match versus lemma identity

The accent-folded `πολιτει` check confirms a visible stem, not a morphological parse. Manual reading is needed to distinguish noun forms, related derivatives, apparatus entries, and textual variants.

### Passage context versus complete argument

Scaife passage units and centered windows may omit preceding or following argumentation. Some texts include critical apparatus that can dominate the returned passage.

### Category versus dictionary sense

An LLM-generated category is an analytical convenience. It is not automatically a lexicographic sense, and one passage can support overlapping readings.

### Valid citation versus supported claim

The audit proves that a cited URN was supplied. It does not prove the cited Greek entails the category claim. Read the context and use the critic/human review.

## 29 - Extend the research design <a class="anchor" id="follow-up"></a>
##### [Back to ToC](#TOC)

Strong follow-up work includes:

- collect additional pages for each strategy and report saturation by work;
- run separate exact-form searches for `πολιτεία`, `πολιτείας`, `πολιτείᾳ`, `πολιτείαν`, `πολιτειῶν`, and related forms;
- inspect Scaife reader-search results within individual works that dominate candidate counts;
- retrieve adjacent passage units for contexts truncated at a citation boundary;
- manually tag each occurrence for morphology, syntactic role, metaphorical/literal status, and proposed sense;
- compare manually tagged categories with the LLM proposal;
- calculate work-normalized frequencies only after collecting a genuinely exhaustive occurrence list;
- search related vocabulary such as `πόλις`, `πολίτης`, `πολιτεύομαι`, `νόμος`, `οἰκονομία`, and `βίος`;
- compare Philo with Josephus, Plato, Aristotle, or Stoic corpora using separate, equally documented packets;
- consult critical editions, translations, lexica, and scholarship outside this notebook before making historical claims.

## 30 - Privacy, cost, and reproducibility <a class="anchor" id="privacy-cost"></a>
##### [Back to ToC](#TOC)

When a model pass is enabled, OpenRouter receives:

- the system instructions;
- the research prompt;
- the full compact evidence packet, including Greek contexts and URNs;
- for later passes, the generated analysis and critique.

It does not receive the API key inside the prompt. The key is used only in the authorization header.

Each additional pass resends substantial evidence and can increase cost. Record returned usage fields and keep synthesis, critique, and revision independently optional.

For reproducibility, retain:

- search plan and page limits;
- execution date;
- scope and passage URNs;
- sampling policy;
- evidence packet JSON;
- requested and resolved model IDs;
- raw model output, parsed JSON, citation audit, and usage;
- critique and revision as separate artifacts.

Review and clear credentialed outputs before committing. Scan the notebook for a full key pattern if it has been run with credentials.

## 31 - Troubleshooting reference <a class="anchor" id="troubleshooting"></a>
##### [Back to ToC](#TOC)

| Symptom | Likely cause and response |
|---|---|
| Scaife metadata does not confirm `tlg0018` | Upstream route or identifier changed; stop and re-establish scope |
| Author discovery omits Philo | Refresh metadata; current servers merge CTS and Scaife inventories and should return `tlg0018` |
| Lemma search returns zero | Treat as an indexing observation; use exact/wildcard strategies and document the discrepancy |
| Search totals are large but candidate pool is small | Only bounded pages were collected |
| One work dominates candidates | Ranking/page bias; inspect coverage and use stratified sampling |
| Wildcard candidate lacks target stem in fetched text | Snippet/index mismatch or related-form match; quality filter excludes it |
| Passage text is mostly apparatus | Centered context may help; otherwise retrieve adjacent context or consult an edition |
| Evidence packet exceeds the character budget | Reduce passages/context length or split the research question |
| Model slug is unavailable | Select a current OpenRouter model and rerun catalog preflight |
| HTTP 401/403 | Check or rotate the API key |
| HTTP 402/429 | Credits, pricing, rate limit, or provider capacity; do not retry blindly |
| Model returns prose instead of JSON | Inspect raw content; strengthen JSON instruction or use a model advertising `response_format` |
| Citation audit finds an unknown URN | Reject or revise the output; do not silently normalize a guessed identifier |
| Audit passes but a claim still looks wrong | Membership is not entailment; inspect Greek context and run human/critic review |
| Critic cites new passages | Reject those citations; the critic must use the same packet |
| Revision changes the research scope | Reject it and rerun with stronger sample-bounded instructions |

For a reproducible bug report, include the search plan, page limits, scope URN, evidence packet validation, model response ID, resolved model, usage, raw-output prefix, and citation audit.

## 32 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Use the earlier notebooks as references for each component:

- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) — search-hit anatomy and CTS/Scaife URN mapping;
- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) — complete tool schemas and operational notes;
- [`06_openrouter_llm_mcp_interaction.ipynb`](06_openrouter_llm_mcp_interaction.ipynb) — model protocols, limits, usage, and security;
- [`07_mcp_advanced_search_options.ipynb`](07_mcp_advanced_search_options.ipynb) — normalization, operators, pagination, and scoping;
- [`08_mcp_cache_and_search_tools.ipynb`](08_mcp_cache_and_search_tools.ipynb) — reader search, highlights, Scaife retrieval, and evidence packets.

Suggested exercises:

1. change page limits and compare candidate/work saturation;
2. replace wildcard search with separate exact inflected-form searches;
3. alter the sampling policy and compare model categories;
4. reduce context windows and test whether the critique detects missing context;
5. manually label five passages, then compare human and model categories;
6. run the same packet through two models and compare citation audits and overreach;
7. adapt the complete pipeline to another author, term, or conceptual field.

## 33 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook is grounded in:

- the local MCP implementation in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
- live schemas returned by FastMCP `client.list_tools()`;
- project guidance in the [README](../README.md), [`docs/architecture.md`](../docs/architecture.md), and [`docs/enduser.md`](../docs/enduser.md);
- search behavior tested in [`tests/test_greek_query_normalization.py`](../tests/test_greek_query_normalization.py) and [`tests/test_exploration_tools.py`](../tests/test_exploration_tools.py);
- [FastMCP](https://github.com/jlowin/fastmcp) for the in-process MCP client;
- the [Scaife Viewer](https://scaife.perseus.org/) search and library services for Philo scope, search hits, and Greek passage text;
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) CTS inventory used for comparison by some discovery tools;
- the [OpenRouter Free Models Router](https://openrouter.ai/openrouter/free) for capability-aware routing across currently available free models;
- the [OpenRouter quickstart](https://openrouter.ai/docs/quickstart), [API reference](https://openrouter.ai/docs/api/reference/overview), and [models endpoint](https://openrouter.ai/api/v1/models) for optional model synthesis and usage metadata.

Search counts, ranking, indexed texts, passage content, model availability, pricing, and generated analysis are live and mutable. The evidence packet records observations from one bounded run rather than permanent facts about the services or corpus.

## 34 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. Recommended installation:

```bash
pip install -e .
```

or:

```bash
uv sync
```

Principal third-party libraries:

- `fastmcp>=2.12.0` for the MCP client and local server;
- `httpx>=0.27.0` for Scaife and optional OpenRouter requests;
- `python-dotenv>=1.0.0` for environment configuration;
- Jupyter/IPython for asynchronous cells and Markdown rendering.

`collections`, `datetime`, `getpass`, `html`, `importlib`, `json`, `os`, `pathlib`, `re`, `sys`, and `unicodedata` are standard-library modules.

An OpenRouter API key is required only when an optional synthesis, critic, or revision pass is enabled.

## 35 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>2.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>